# Preprocessing

In [1]:
import pandas as pd

products = pd.read_csv("/kaggle/input/datasets/sadmanshafiq1611/product-details/product_details.csv")

print(products.head())
print(products.shape)
print(products.info())
print(products.describe())



                                    URL  \
0  https://www.amazon.com/dp/B0GJ5KNC27   
1  https://www.amazon.com/dp/B0FRN2Z4JF   
2  https://www.amazon.com/dp/B09C13PZX7   
3  https://www.amazon.com/dp/B0G12HGHGM   
4  https://www.amazon.com/dp/B0BX52C4LN   

                                               Title       Brand Rating  \
0  GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...   GravaStar    4.5   
1  GravaStar Mercury M1 Pro Wireless Gaming Mouse...   GravaStar    4.5   
2  Razer Basilisk V3 Customizable RGB Wired Ergon...       Razer    4.6   
3  Logitech G PRO X2 SUPERSTRIKE Wireless Gaming ...  Logitech G    4.5   
4  HyperX Pulsefire Haste 2 – Wireless Gaming Mou...      HyperX    4.1   

    Price Review_Count                                          Image_URL  \
0   71.95        2,070  https://m.media-amazon.com/images/I/61M5yEjuIF...   
1   89.96        1,485  https://m.media-amazon.com/images/I/51BhB+GhmQ...   
2   34.98       10,516  https://m.media-amazon.com/images/I

In [2]:
print(len(products))
products = products[products["Image_URL"] != "not available"]

print(len(products))

1162
1162


In [3]:
import os
image_rows = []
for index, row in products.iterrows():
    asin = row["URL"][-10:]
    
    image_path = os.path.join("/kaggle/input/datasets/sadmanshafiq1611/amazon-images/images/", asin + ".jpg")
    
    if not os.path.exists(image_path):
        continue
        
    image_rows.append({
        "ASIN": asin,
        "Title": row["Title"],
        "Image_Path": image_path,
        "Category": row["Category"],
        "Brand": row["Brand"],
    })

images_df = pd.DataFrame(image_rows)

print("Products with images:", len(images_df))
print("distinct categories:", images_df["Category"].nunique())


Products with images: 1162
distinct categories: 172


In [4]:
# from PIL import Image
# import matplotlib.pyplot as plt

# image_path = images_df.iloc[0]["Image_Path"]

# image = Image.open(image_path)

# plt.imshow(image)
# plt.show()

In [5]:
import torch

from transformers import AutoImageProcessor, AutoModel



device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model = model.to(device)
model.eval()

print(device)

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

cuda


In [6]:
image_paths = images_df["Image_Path"]
print(image_paths)

0       /kaggle/input/datasets/sadmanshafiq1611/amazon...
1       /kaggle/input/datasets/sadmanshafiq1611/amazon...
2       /kaggle/input/datasets/sadmanshafiq1611/amazon...
3       /kaggle/input/datasets/sadmanshafiq1611/amazon...
4       /kaggle/input/datasets/sadmanshafiq1611/amazon...
                              ...                        
1157    /kaggle/input/datasets/sadmanshafiq1611/amazon...
1158    /kaggle/input/datasets/sadmanshafiq1611/amazon...
1159    /kaggle/input/datasets/sadmanshafiq1611/amazon...
1160    /kaggle/input/datasets/sadmanshafiq1611/amazon...
1161    /kaggle/input/datasets/sadmanshafiq1611/amazon...
Name: Image_Path, Length: 1162, dtype: object


In [42]:
from PIL import Image
from tqdm.auto import tqdm
import numpy as np

BATCH_SIZE = 32

embeddings = []
valid_paths = []

for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):

    batch_paths = image_paths[i:i + BATCH_SIZE]

    images = []

    for path in batch_paths:
        try:
            image = Image.open(path).convert("RGB")
            images.append(image)

        except Exception as e:
            print("Error:", path, e)


    inputs = processor(images=images,return_tensors="pt")

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs)

    batch_embeddings = outputs.last_hidden_state[:, 0, :]

    embeddings.append(
        batch_embeddings.cpu().numpy()
    )



embeddings = np.vstack(embeddings)

print("embedding shape:", embeddings.shape)

  0%|          | 0/37 [00:00<?, ?it/s]

embedding shape: (1162, 768)


In [52]:
# np.save("/kaggle/working/embeddings.npy", embeddings)
embeddings = np.load("/kaggle/working/embeddings.npy")

In [ ]:
import hdbscan
import numpy as np
from sklearn.metrics import silhouette_score

results = []

min_cs = [3, 5, 10, 15, 20, 30]
min_sv = [1, 3, 5, 10]

for min_cluster_size in min_cs:
    for min_samples in min_sv:

        clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,min_samples=min_samples,metric="euclidean")

        labels = clusterer.fit_predict(embeddings)

        not_noise = labels != -1

        filtered_embeddings = embeddings[not_noise]
        filtered_labels = labels[not_noise]

        cluster_no = len(set(filtered_labels))

        score = silhouette_score(filtered_embeddings,filtered_labels)

        results.append({
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "silhouette_score": score,
            "no. of clusters": cluster_no,
            "noise": np.sum(labels == -1)
        })

        print(f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, score={score:.4f}, clusters={cluster_no},noise={np.sum(labels == -1)}")

In [ ]:
results_df = pd.DataFrame(results)

best = results_df.loc[results_df["silhouette_score"].idxmax()]

print("best:")
print(best)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=3,min_samples=10,metric="euclidean")

labels = clusterer.fit_predict(embeddings)

In [55]:

unique, counts = np.unique(labels, return_counts=True)

for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} images")

Cluster -1: 325 images
Cluster 0: 57 images
Cluster 1: 31 images
Cluster 2: 66 images
Cluster 3: 54 images
Cluster 4: 17 images
Cluster 5: 25 images
Cluster 6: 63 images
Cluster 7: 19 images
Cluster 8: 25 images
Cluster 9: 31 images
Cluster 10: 27 images
Cluster 11: 15 images
Cluster 12: 63 images
Cluster 13: 26 images
Cluster 14: 11 images
Cluster 15: 35 images
Cluster 16: 34 images
Cluster 17: 59 images
Cluster 18: 30 images
Cluster 19: 23 images
Cluster 20: 23 images
Cluster 21: 55 images
Cluster 22: 31 images
Cluster 23: 17 images
